In [ ]:
from GCMC import *
import pandas as pd
import psycopg2
from sklearn.preprocessing import LabelEncoder
import numpy as np
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.utils.class_weight import compute_class_weight
from torch_optimizer import Ranger
from sklearn.metrics import confusion_matrix

In [3]:
# Connect PostgreSQL
conn = psycopg2.connect(
    dbname="mydb",
    user="user",
    password="pass",
    host="my_postgres",
    port="5432"
)

# Import data using Pandas
df = pd.read_sql(
    """SELECT index
            , recipe_code
            , recipe_name
            , user_id
            , stars
            , agg_rating
            , user_reputation
            , food_category
            , feature 
       FROM new_review""", conn)
print(df.head())

conn.close()


   index  recipe_code         recipe_name         user_id  stars  agg_rating  \
0      0        14299  Creamy White Chili  u_9iFLIhMa8QaG      5    3.557082   
1      1        14299  Creamy White Chili  u_Lu6p25tmE77j      5    4.661161   
2      2        14299  Creamy White Chili  u_s0LwgpZ8Jsqq      5    4.518386   
3      3        14299  Creamy White Chili  u_fqrybAdYjgjG      0    1.296705   
4      4        14299  Creamy White Chili  u_XXWKwVhKZD69      0    1.762641   

   user_reputation food_category                           feature  
0                1    Soup/Chili  #creamy, #white, #chili, #hearty  
1               50    Soup/Chili  #creamy, #white, #chili, #hearty  
2               10    Soup/Chili  #creamy, #white, #chili, #hearty  
3                1    Soup/Chili  #creamy, #white, #chili, #hearty  
4               10    Soup/Chili  #creamy, #white, #chili, #hearty  


/tmp/ipykernel_3266/4105608882.py:11: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(


In [46]:
def split_data(df):
    
    total_indices = np.arange(len(df))
    np.random.shuffle(total_indices)

    train_size = int(len(total_indices) * 0.7)
    valid_size = int(len(total_indices) * 0.15)  # validation 15%
    test_size = len(total_indices) - train_size - valid_size  # 나머지는 test

    train_indices = total_indices[:train_size]
    valid_indices = total_indices[train_size:train_size + valid_size]
    test_indices = total_indices[train_size + valid_size:]
    
    train_df = df.iloc[train_indices]
    test_df = df.iloc[test_indices]
    valid_df = df.iloc[valid_indices] 
    
    return train_df, test_df, valid_df

def safe_transform(encoder, values, unknown_val=-1):
    known = set(encoder.classes_)
    class_to_index = {cls: i for i, cls in enumerate(encoder.classes_)}
    return [class_to_index[v] if v in known else unknown_val for v in values]
    
def preprocessing(df):
    
    global num_users, num_items
    
    user_encoder = LabelEncoder().fit(df['user_id'])
    item_encoder = LabelEncoder().fit(df['recipe_code'])

    df['user_idx'] = user_encoder.transform(df['user_id'])
    df['item_idx'] = item_encoder.transform(df['recipe_code'])

    train_df, test_df, valid_df = split_data(df) 
    
    num_users = len(user_encoder.classes_)
    num_items = len(item_encoder.classes_)
    
    # train
    user_reputation_train = train_df.groupby('user_idx')['user_reputation'].mean().reindex(range(num_users)).fillna(0)
    u_feat_side_train = torch.tensor(user_reputation_train.values).unsqueeze(1)  # shape: [num_users, 1]
    u_feat_side_train = u_feat_side_train.to(torch.float32)
    
    # valid
    user_reputation_valid = valid_df.groupby('user_idx')['user_reputation'].mean().reindex(range(num_users)).fillna(0)
    u_feat_side_valid = torch.tensor(user_reputation_valid.values).unsqueeze(1)  # shape: [num_users, 1]
    u_feat_side_valid = u_feat_side_valid.to(torch.float32)  
    
    # test
    user_reputation_test = test_df.groupby('user_idx')['user_reputation'].mean().reindex(range(num_users)).fillna(0)
    u_feat_side_test = torch.tensor(user_reputation_test.values).unsqueeze(1)  # shape: [num_users, 1]
    u_feat_side_test = u_feat_side_test.to(torch.float32)
     
    cat_encoder = OneHotEncoder()
        
    mlb = MultiLabelBinarizer()

    
    # create category side info based on total items
    cat_onehot = cat_encoder.fit_transform(df[['food_category']])
    item_cat = pd.DataFrame(cat_onehot.toarray()).groupby(df['item_idx']).mean()
    item_cat = item_cat.reindex(range(num_items)).fillna(0)
    v_cat_side = torch.tensor(item_cat.values, dtype=torch.float32)

    # binarize feature list based on total items
    df['feature_list'] = df['feature'].fillna("").apply(lambda x: x.split(', ') if x else [])
    tag_binary = mlb.fit_transform(df['feature_list'])
    item_feat = pd.DataFrame(tag_binary).groupby(df['item_idx']).mean()
    item_feat = item_feat.reindex(range(num_items)).fillna(0)
    v_tag_side = torch.tensor(item_feat.values, dtype=torch.float32)

    # final item side info
    v_feat_side = torch.cat([v_cat_side, v_tag_side], dim=1)
    v_feat_side = v_feat_side.to(torch.float32)

    
    return num_users, num_items, u_feat_side_train, v_feat_side, u_feat_side_test, u_feat_side_valid, train_df, test_df, valid_df 




In [109]:
# list of sparse adjacency matrix, indicating user_item relationship by stars
# support[i] indicates user-
def make_support_matrix(df, num_users, num_items, num_classes):
    supports = []
    for rating in range(1, num_classes + 1):
        mask = df['stars'] == rating
        rows = df[mask]['user_idx'].values
        cols = df[mask]['item_idx'].values
        values = np.ones(len(rows))

        coo = torch.sparse_coo_tensor(
            indices=torch.tensor([rows, cols]),
            values=torch.tensor(values, dtype=torch.float32),
            size=(num_users, num_items)
        )
        supports.append(coo.coalesce())
    return supports


In [110]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class FocalLoss(nn.Module):
    def __init__(self, alpha=None, gamma=2.0, reduction='mean'):
        super(FocalLoss, self).__init__()
        self.alpha = alpha  # Type of tensor : [C] or None
        self.gamma = gamma
        self.reduction = reduction

    def forward(self, inputs, targets):
        """
        inputs: (N, C) - logit (not apply softmax)
        targets: (N,) - constant label
        """
        ce_loss = F.cross_entropy(inputs, targets, reduction='none', weight=self.alpha)
        pt = torch.exp(-ce_loss)  
        focal_loss = ((1 - pt) ** self.gamma) * ce_loss

        if self.reduction == 'mean':
            return focal_loss.mean()
        elif self.reduction == 'sum':
            return focal_loss.sum()
        else:
            return focal_loss


In [ ]:
def train_model(df, num_classes, num_basis_functions, hidden_dims = [72, 32], accum='sum', model_ty='classify',
                self_connections=False, dropout=0.5, input_dim=128, epoch=100, model_save=True,
                opti='adam', loss_ty='NLL', patience=5):
    
    num_users, num_items, u_feat_side_train, v_feat_side, u_feat_side_test, u_feat_side_valid, train_df, test_df, valid_df = preprocessing(df)
    support = make_support_matrix(train_df, num_users, num_items, 6)
    support_t = [s.transpose(0, 1) for s in support]

    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

    # Define Loss
    if model_ty == 'classify':
        manual_weights = {0:1.3, 1:4, 2:4, 3:4, 4:2, 5:1} 
        class_weights = compute_class_weight(class_weight='balanced', classes=np.arange(num_classes), y=train_df['stars'].values)
        alpha = torch.FloatTensor(class_weights).to(device) 
        if loss_ty == 'NLL':
            loss_fn = nn.NLLLoss(weight=alpha)
        elif loss_ty == 'Focal':
            loss_fn = FocalLoss(alpha=alpha, gamma=2.0)
    elif model_ty == 'regression':
        loss_fn = nn.MSELoss() if loss_ty == 'MSE' else nn.SmoothL1Loss()
            
    model = RecommenderSideInfoGAE(
        input_dim=input_dim,
        feat_hidden_dim=u_feat_side_train.shape[1],
        hidden_dims=hidden_dims,
        num_support=len(support),
        num_classes=num_classes,
        num_basis_functions=num_basis_functions,
        num_users=num_users,
        num_items=num_items,
        u_num_side_features=u_feat_side_train.shape[1],
        v_num_side_features=v_feat_side.shape[1],
        accum=accum,
        self_connections=self_connections,
        dropout=dropout,
        model_ty=model_ty
    ).to(device)
    
    if opti == 'adam':
        optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
    elif opti =='adamw':
        optimizer = torch.optim.AdamW(model.parameters(), lr=0.001) 
    elif opti == 'ranger':
        optimizer = Ranger(model.parameters(), lr=0.001)

    # variables for Early Stopping
    best_val_loss = float('inf')
    best_model_state = None
    counter = 0

    for ep in range(epoch):
        model.train()
        shuffled = train_df.sample(frac=1).reset_index(drop=True)
        batch_size = 512
        
        target_val = 'stars' if model_ty == 'classify' else 'agg_rating'
        data_ty = torch.long if model_ty == 'classify' else torch.float32

        for i in range(0, len(shuffled), batch_size):
            batch_df = shuffled.iloc[i:i+batch_size]
            u_idx = torch.tensor(batch_df['user_idx'].values, dtype=torch.long).to(device)
            v_idx = torch.tensor(batch_df['item_idx'].values, dtype=torch.long).to(device)
            labels = torch.tensor(batch_df[target_val].values, dtype=data_ty).to(device)

            output = model(
                u_feat_side_train.to(device),
                v_feat_side.to(device),
                [s.to(device) for s in support],
                [s.to(device) for s in support_t],
                u_idx, v_idx
            )

            loss = loss_fn(output, labels)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

        # Validation Step
        model.eval()
        with torch.no_grad():
            u_idx_val = torch.tensor(valid_df['user_idx'].values, dtype=torch.long).to(device)
            v_idx_val = torch.tensor(valid_df['item_idx'].values, dtype=torch.long).to(device)
            val_labels = torch.tensor(valid_df[target_val].values, dtype=data_ty).to(device)

            val_output = model(
                u_feat_side_valid.to(device),
                v_feat_side.to(device),
                [s.to(device) for s in support],
                [s.to(device) for s in support_t],
                u_idx_val, v_idx_val
            )

            val_loss = loss_fn(val_output, val_labels)

        print(f"Epoch {ep+1} | Train Loss: {loss.item():.4f} | Val Loss: {val_loss.item():.4f}")

        # check Early Stopping
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_model_state = model.state_dict()
            counter = 0
        else:
            counter += 1
            if counter >= patience:
                print(f"Early stopping triggered at epoch {ep+1}")
                break

    print(f"Best model's val loss is {best_val_loss:.4f}")

    if model_save:
        torch.save(best_model_state, "/root/profile/tech blog/model_weight/best_model.pth")  

    model.load_state_dict(best_model_state)  # load best model
    return model


In [166]:
model = train_model(df, num_classes = 6, num_basis_functions = 3, accum='stack', opti='ranger', loss_ty = 'Focal', patience=10)

Epoch 1 | Train Loss: 2.1469 | Val Loss: 1.6345
Epoch 2 | Train Loss: 1.5721 | Val Loss: 1.6314
Epoch 3 | Train Loss: 1.4403 | Val Loss: 1.6289
Epoch 4 | Train Loss: 1.5185 | Val Loss: 1.6262
Epoch 5 | Train Loss: 1.2276 | Val Loss: 1.6203
Epoch 6 | Train Loss: 1.2657 | Val Loss: 1.6132
Epoch 7 | Train Loss: 1.1753 | Val Loss: 1.6074
Epoch 8 | Train Loss: 1.2533 | Val Loss: 1.5997
Epoch 9 | Train Loss: 1.2399 | Val Loss: 1.5940
Epoch 10 | Train Loss: 1.0193 | Val Loss: 1.5943
Epoch 11 | Train Loss: 0.9621 | Val Loss: 1.5836
Epoch 12 | Train Loss: 0.7810 | Val Loss: 1.5958
Epoch 13 | Train Loss: 0.8943 | Val Loss: 1.5971
Epoch 14 | Train Loss: 0.9363 | Val Loss: 1.6062
Epoch 15 | Train Loss: 0.8465 | Val Loss: 1.6054
Epoch 16 | Train Loss: 1.0185 | Val Loss: 1.6217
Epoch 17 | Train Loss: 0.6480 | Val Loss: 1.6304
Epoch 18 | Train Loss: 0.6117 | Val Loss: 1.6361
Epoch 19 | Train Loss: 0.5262 | Val Loss: 1.6451
Epoch 20 | Train Loss: 0.6808 | Val Loss: 1.6668
Epoch 21 | Train Loss: 0.6367

In [66]:
num_users, num_items, u_feat_side_train, v_feat_side, u_feat_side_test, u_feat_side_valid, train_df, test_df, valid_df = preprocessing(df)

In [67]:
support = make_support_matrix(train_df, num_users, num_items, 6)
support_t = [s.transpose(0, 1) for s in support]

In [167]:
u_indices_test = torch.tensor(test_df['user_idx'].values, dtype=torch.long)
v_indices_test = torch.tensor(test_df['item_idx'].values, dtype=torch.long)
labels_test = torch.tensor(test_df['stars'].values , dtype=torch.long)
# labels_test = torch.tensor(test_df['agg_rating'].values , dtype=torch.float32)

# forward
output = model(
    u_feat_side_test, v_feat_side,
    support, support_t,
    u_indices_test, v_indices_test
)

loss_fn = nn.MSELoss()  # 1. Loss 객체 만들고
# loss = loss_fn(output, labels_test)

loss = F.nll_loss(output, labels_test)


In [168]:
result = []
for l in output:
    result.append(torch.where(l == max(l))[0].item())
    
test_df['expectation'] = result
test_df['con_yn'] = test_df.stars == test_df.expectation



/tmp/ipykernel_3266/3158468177.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  test_df['expectation'] = result
/tmp/ipykernel_3266/3158468177.py:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  test_df['con_yn'] = test_df.stars == test_df.expectation


In [ ]:
# when model type is regression
# test_df['expectation'] = output.tolist()

/tmp/ipykernel_3266/1485216898.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  test_df['expectation'] = output.tolist()


In [169]:
cm = confusion_matrix(test_df['stars'], test_df['expectation'])

In [ ]:
cm # sum + adam + NLL 

array([[ 230,    1,    1,   13,    0,    7],
       [  39,    0,    0,    4,    0,    0],
       [  31,    0,    0,    7,    0,    0],
       [  51,    0,    0,   20,    0,    5],
       [ 156,    3,    4,   73,    0,   14],
       [1417,   28,    3,  462,    0,  159]])

In [ ]:
cm # sum + adamw + NLL

array([[ 178,   15,   27,    5,   10,   17],
       [  33,    4,    3,    0,    2,    1],
       [  23,    5,    4,    1,    3,    2],
       [  44,   15,    3,    1,   10,    3],
       [ 125,   28,   23,   13,   51,   10],
       [1158,  275,  171,   63,  295,  107]])

In [ ]:
cm # sum + ranger + NLL

array([[ 230,    2,    5,    2,   12,    1],
       [  39,    1,    0,    0,    2,    1],
       [  31,    1,    3,    0,    1,    2],
       [  51,    2,    7,    3,   11,    2],
       [ 154,   11,   27,   12,   41,    5],
       [1404,   91,  131,  120,  261,   62]])

In [125]:
cm # sum + adam + Focal

array([[ 242,    0,    0,    4,    5,    1],
       [  39,    0,    0,    2,    2,    0],
       [  33,    1,    0,    1,    3,    0],
       [  54,    0,    0,   15,    7,    0],
       [ 168,    6,    0,   28,   48,    0],
       [1602,   40,    0,  231,  193,    3]])

In [135]:
cm # sum + adamw + Focal

array([[ 234,    1,    0,    1,   16,    0],
       [  39,    0,    0,    0,    4,    0],
       [  32,    0,    0,    0,    6,    0],
       [  54,    0,    0,    2,   20,    0],
       [ 166,    3,    0,    0,   81,    0],
       [1575,   11,    0,   21,  461,    1]])

In [140]:
cm # sum + ranger + Focal

array([[ 238,    1,    1,    1,   11,    0],
       [  39,    0,    0,    0,    4,    0],
       [  33,    1,    0,    0,    4,    0],
       [  54,    0,    1,    2,   18,    1],
       [ 167,    3,    2,    5,   72,    1],
       [1569,   29,   15,   30,  422,    4]])

In [145]:
cm # stack + adam + NLL

array([[ 238,    1,    1,    1,   11,    0],
       [  39,    0,    0,    0,    4,    0],
       [  33,    1,    0,    0,    4,    0],
       [  54,    0,    1,    2,   18,    1],
       [ 167,    3,    2,    5,   72,    1],
       [1569,   29,   15,   30,  422,    4]])

In [150]:
cm # stack + adamw + NLL

array([[ 233,    1,    1,    4,    1,   12],
       [  39,    1,    0,    0,    1,    2],
       [  31,    0,    3,    0,    1,    3],
       [  55,    0,    0,   10,    6,    5],
       [ 161,    0,    5,   10,   49,   25],
       [1517,    8,   31,   95,  112,  306]])

In [155]:
cm # stack + ranger + NLL

array([[ 235,    0,    0,    0,    1,   16],
       [  40,    0,    0,    0,    1,    2],
       [  31,    0,    3,    0,    1,    3],
       [  52,    0,    1,    9,    6,    8],
       [ 162,    0,    5,    6,   52,   25],
       [1549,    7,   15,   27,  112,  359]])

In [160]:
cm # stack + adam + Focal

array([[142, 104,   0,   0,   4,   2],
       [ 19,  22,   0,   0,   1,   1],
       [ 20,  12,   3,   0,   3,   0],
       [ 35,  21,   0,   9,  10,   1],
       [ 90,  83,   4,   7,  64,   2],
       [840, 873,  42,  22, 244,  48]])

In [165]:
cm # stack + adamw + Focal

array([[ 249,    1,    0,    0,    2,    0],
       [  39,    1,    1,    0,    2,    0],
       [  33,    0,    2,    1,    2,    0],
       [  60,    1,    0,    9,    6,    0],
       [ 186,    4,    5,    5,   50,    0],
       [1798,   24,   46,   23,  178,    0]])

In [170]:
cm # stack + ranger + Focal

array([[ 244,    0,    1,    3,    4,    0],
       [  41,    1,    0,    0,    1,    0],
       [  32,    0,    3,    0,    3,    0],
       [  56,    0,    1,   10,    8,    1],
       [ 173,    4,    5,   12,   52,    4],
       [1740,   19,   38,   58,  173,   41]])